### Librerias y Variables Globales

In [ ]:
from google.cloud import storage
from google.cloud import bigquery
import pandas as pd
import pytz
import io
import re
from datetime import datetime, timedelta

In [ ]:
client = storage.Client()
clientBQ = bigquery.Client()

In [ ]:
Zona = pytz.timezone('America/Lima')
peru_time = datetime.now(Zona)
peru_time = peru_time - timedelta(days=1) ## Dia anterior
var_fecha = peru_time.strftime('%Y-%m-%d')
peru_time_ini = peru_time - timedelta(days=10) ## 10 Dias anterior
var_fecha_2 = peru_time_ini.strftime('%Y-%m-%d')

In [ ]:
var_fecha_2,var_fecha

('2026-03-16', '2026-03-26')

In [ ]:
# # Parámetros de fecha
#fecha_inicial = datetime.strptime("2025-01-01", "%Y-%m-%d")
#fecha_final = datetime.strptime("2026-01-20", "%Y-%m-%d")

In [ ]:
fecha_inicial = datetime.strptime(var_fecha_2, "%Y-%m-%d")
fecha_final = datetime.strptime(var_fecha, "%Y-%m-%d")

In [ ]:
var_fecha_ini = fecha_inicial.strftime('%Y-%m-%d')
var_fecha_fin = fecha_final.strftime('%Y-%m-%d')

In [ ]:
bucket_name = 'prd-ingesta-izipay-fx32'

In [ ]:
#@title Variable Dataset & Tables { run: "auto", display-mode: "form" }
var_project_storage = "prd-izipay-data-storage-pv" #@param {type:"string"}
var_project_sensitive = "prd-izipay-data-sensitive" #@param {type:"string"}
var_project_operation = "prd-izipay-data-operation" #@param {type:"string"}

### BBVA PMP

In [ ]:
#@title Variables Banco { run: "auto", display-mode: "form" }
var_banco = "bbva" #@param {type:"string"}
var_banco_cod = "MCPR345" #@param {type:"string"}
var_table = "rpta_rechazo_bbva_pmp" #@param {type:"string"}

#### Variables Reproceso

In [ ]:
# Parámetros de fecha
# fecha_inicial = datetime.strptime("2025-09-28", "%Y-%m-%d")
# fecha_final = datetime.strptime("2025-11-26", "%Y-%m-%d")

In [ ]:
fecha_inicial,fecha_final

(datetime.datetime(2026, 3, 16, 0, 0), datetime.datetime(2026, 3, 26, 0, 0))

#### Carga Archivos

In [ ]:
# Crear cliente de Cloud Storage
bucket = client.bucket(bucket_name)

In [ ]:
# Lista para almacenar los DataFrames
dataframes = []

In [ ]:
fecha_actual = fecha_inicial
while fecha_actual <= fecha_final:
    fecha_str = fecha_actual.strftime("%Y%m%d")
    anho_str = fecha_actual.strftime("%Y")
    mes_str = fecha_actual.strftime("%m")
    dia_str = fecha_actual.strftime("%d")

    folder_path = f'Raw/Data_Entry/Finanzas/{var_banco}/{anho_str}/{mes_str}/{dia_str}'
    # --filename = f'MCPR345_{fecha_str}_190301_Rep_operaciones_rechazadas_BBVA_PMP.txt'

    prefijo = f"{folder_path}/{var_banco_cod}_{fecha_str}_"
    blobs = bucket.list_blobs(prefix=prefijo)
    blobs_filter = [blob for blob in blobs if 'PMP' in blob.name]
    for blob in blobs_filter:

    # Ruta completa dentro del bucket
    # --blob_name = f'{folder_path}/{filename}'
    # print({blob_name})
    # --blob = bucket.blob(blob_name)

      if blob.exists():
              # Leer el contenido del archivo
              # --file_content = blob.download_as_text()
              file_content = blob.download_as_string()

              # Dividir por líneas
              lines = file_content.splitlines()

              # Eliminar las primeras 5 líneas y la línea 7 (índice 6 después de eliminar 5)
              lines_filtered = lines[5:]  # eliminar primeras 5

              # Convertir a un archivo de tipo buffer (simula un archivo real)
              lines_filtered_str = [line.decode("latin1") for line in lines_filtered]

              lines_fixed = [line.replace("A  TIPO DOC.","A     TIPO DOC.") for line in lines_filtered_str]

              buffer = io.StringIO('\n'.join(lines_fixed))
              # --buffer = io.StringIO('\n'.join(lines_filtered))
              df = pd.read_csv(buffer, sep=r'\s{4,}', engine='python', encoding='utf-8', dtype=str)
              df = df[df.isnull().sum(axis=1) < 5].reset_index(drop=True)
              df.columns = [
                  "comercio",
                  "fecha_apertura",
                  "tipo_documento",
                  "nro_documento",
                  "razon_social",
                  "nro_cuenta",
                  "banco",
                  "moneda",
                  "importe",
                  "archivo",
                  "motivo_rechazo"
              ]
              # Agregar las columnas con los valores extraídos del nombre del archivo
              df['process_date'] = fecha_str  # La fecha extraída del nombre del archivo
              dataframes.append(df)
              print(f"Carga Exitosa!!!: {blob}")
      else:
          print(f"Archivo no encontrado en GCS: {blob}")

    fecha_actual += timedelta(days=1)

Carga Exitosa!!!: <Blob: prd-ingesta-izipay-fx32, Raw/Data_Entry/Finanzas/bbva/2026/03/16/MCPR345_20260316_190001_Rep_operaciones_rechazadas_BBVA_PMP.txt, 1773760513182204>
Carga Exitosa!!!: <Blob: prd-ingesta-izipay-fx32, Raw/Data_Entry/Finanzas/bbva/2026/03/17/MCPR345_20260317_190301_Rep_operaciones_rechazadas_BBVA_PMP.txt, 1773846922709306>
Carga Exitosa!!!: <Blob: prd-ingesta-izipay-fx32, Raw/Data_Entry/Finanzas/bbva/2026/03/18/MCPR345_20260318_190201_Rep_operaciones_rechazadas_BBVA_PMP.txt, 1773933312889165>
Carga Exitosa!!!: <Blob: prd-ingesta-izipay-fx32, Raw/Data_Entry/Finanzas/bbva/2026/03/19/MCPR345_20260319_190201_Rep_operaciones_rechazadas_BBVA_PMP.txt, 1774019712548717>
Carga Exitosa!!!: <Blob: prd-ingesta-izipay-fx32, Raw/Data_Entry/Finanzas/bbva/2026/03/20/MCPR345_20260320_190301_Rep_operaciones_rechazadas_BBVA_PMP.txt, 1774106111697672>
Carga Exitosa!!!: <Blob: prd-ingesta-izipay-fx32, Raw/Data_Entry/Finanzas/bbva/2026/03/23/MCPR345_20260323_190301_Rep_operaciones_recha

In [ ]:
dataframes

[Empty DataFrame
 Columns: [comercio, fecha_apertura, tipo_documento, nro_documento, razon_social, nro_cuenta, banco, moneda, importe, archivo, motivo_rechazo, process_date]
 Index: [],
 Empty DataFrame
 Columns: [comercio, fecha_apertura, tipo_documento, nro_documento, razon_social, nro_cuenta, banco, moneda, importe, archivo, motivo_rechazo, process_date]
 Index: [],
 Empty DataFrame
 Columns: [comercio, fecha_apertura, tipo_documento, nro_documento, razon_social, nro_cuenta, banco, moneda, importe, archivo, motivo_rechazo, process_date]
 Index: [],
 Empty DataFrame
 Columns: [comercio, fecha_apertura, tipo_documento, nro_documento, razon_social, nro_cuenta, banco, moneda, importe, archivo, motivo_rechazo, process_date]
 Index: [],
 Empty DataFrame
 Columns: [comercio, fecha_apertura, tipo_documento, nro_documento, razon_social, nro_cuenta, banco, moneda, importe, archivo, motivo_rechazo, process_date]
 Index: [],
 Empty DataFrame
 Columns: [comercio, fecha_apertura, tipo_documento, 

In [ ]:
if len(dataframes) == 0:
    df_final = pd.DataFrame()  # Devuelve un DataFrame vacío y continúa
elif len(dataframes) == 1:
    df_final = dataframes[0]
else:
    df_final = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_final['process_date'] = pd.to_datetime(df_final['process_date'], format='%Y%m%d')

#### Carga a Bigquery

In [ ]:
df_final.to_gbq(destination_table=f'raw_stage_dataentry_finanzas.{var_table}',
          project_id = var_project_storage,
          if_exists='replace')  # Opciones: 'fail', 'replace', 'append'

/tmp/ipykernel_8858/1156659946.py:1: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_final.to_gbq(destination_table=f'raw_stage_dataentry_finanzas.{var_table}',


#### Ejecución SP

In [ ]:
# Llama al procedimiento almacenado con los parámetros
query = f"CALL `{var_project_operation}.raw_stage_dataentry_finanzas.prc_load_table_rpta_rechazo_bbva_pmp`('{var_project_storage}','{var_project_sensitive}','{var_fecha_ini}','{var_fecha_fin}');"
job = clientBQ.query(query)
# Espera a que la consulta se complete
results = job.result()

BadRequest: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/prd-izipay-data-operation/queries/5fa6f0c7-2356-4f35-af00-88c032513c43?maxResults=0&location=US&prettyPrint=false: Access Denied: BigQuery BigQuery: User has neither fine-grained reader nor masked get permission to get data protected by policy tag "Izi_Politica_Mask : Alta" on columns prd-izipay-data-sensitive.secure_secrets.config_protected_data.constant, prd-izipay-data-sensitive.secure_secrets.config_protected_data.key. at [prd-izipay-data-operation.raw_stage_dataentry_finanzas.prc_load_table_rpta_rechazo_bbva_pmp:2:5]

Location: US
Job ID: 5fa6f0c7-2356-4f35-af00-88c032513c43


### BBVA IZIPAY

In [ ]:
#@title Variables Banco y Tabla { run: "auto", display-mode: "form" }
var_banco = "bbva" #@param {type:"string"}
var_banco_cod = "MCPR345" #@param {type:"string"}
var_table = "rpta_rechazo_bbva_izipay" #@param {type:"string"}

#### Variables Reproceso

In [ ]:
# Parámetros de fecha
# fecha_inicial = datetime.strptime("2025-09-28", "%Y-%m-%d")
# fecha_final = datetime.strptime("2025-11-26", "%Y-%m-%d")

In [ ]:
fecha_inicial,fecha_final

#### Carga Archivos

In [ ]:
# Crear cliente de Cloud Storage
bucket = client.bucket(bucket_name)

In [ ]:
# Lista para almacenar los DataFrames
dataframes = []

In [ ]:
fecha_actual = fecha_inicial
while fecha_actual <= fecha_final:
    fecha_str = fecha_actual.strftime("%Y%m%d")
    anho_str = fecha_actual.strftime("%Y")
    mes_str = fecha_actual.strftime("%m")
    dia_str = fecha_actual.strftime("%d")

    folder_path = f'Raw/Data_Entry/Finanzas/{var_banco}/{anho_str}/{mes_str}/{dia_str}'
    # --filename = f'MCPR345_{fecha_str}_190301_Rep_operaciones_rechazadas_BBVA_PMP.txt'

    prefijo = f"{folder_path}/{var_banco_cod}_{fecha_str}_"
    blobs = bucket.list_blobs(prefix=prefijo)
    blobs_filter = [blob for blob in blobs if 'IZIPAY' in blob.name]
    for blob in blobs_filter:

    # Ruta completa dentro del bucket
    # --blob_name = f'{folder_path}/{filename}'
    # print({blob_name})
    # --blob = bucket.blob(blob_name)

      if blob.exists():
              # Leer el contenido del archivo
              # --file_content = blob.download_as_text()
              file_content = blob.download_as_string()

              # Dividir por líneas
              lines = file_content.splitlines()

              # Eliminar las primeras 5 líneas y la línea 7 (índice 6 después de eliminar 5)
              lines_filtered = lines[5:]  # eliminar primeras 5

              # Convertir a un archivo de tipo buffer (simula un archivo real)
              lines_filtered_str = [line.decode("latin1") for line in lines_filtered]

              lines_fixed = [line.replace("A  TIPO DOC.","A     TIPO DOC.") for line in lines_filtered_str]

              buffer = io.StringIO('\n'.join(lines_fixed))
              # --buffer = io.StringIO('\n'.join(lines_filtered))
              df = pd.read_csv(buffer, sep=r'\s{4,}', engine='python', encoding='utf-8', dtype=str)
              df = df[df.isnull().sum(axis=1) < 5].reset_index(drop=True)
              df.columns = [
                  "comercio",
                  "fecha_apertura",
                  "tipo_documento",
                  "nro_documento",
                  "razon_social",
                  "nro_cuenta",
                  "banco",
                  "moneda",
                  "importe",
                  "archivo",
                  "motivo_rechazo"
              ]
              # Agregar las columnas con los valores extraídos del nombre del archivo
              df['process_date'] = fecha_str  # La fecha extraída del nombre del archivo
              dataframes.append(df)
              print(f"Carga Exitosa!!!: {blob}")
      else:

          print(f"Archivo no encontrado en GCS: {blob}")

    fecha_actual += timedelta(days=1)

In [ ]:
dataframes

In [ ]:
if len(dataframes) == 0:
    df_final = pd.DataFrame()  # Devuelve un DataFrame vacío y continúa
elif len(dataframes) == 1:
    df_final = dataframes[0]
else:
    df_final = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_final['process_date'] = pd.to_datetime(df_final['process_date'], format='%Y%m%d')

In [ ]:
print(df_final.dtypes)

#### Carga a Bigquery

In [ ]:
df_final.to_gbq(destination_table=f'raw_stage_dataentry_finanzas.{var_table}',
          project_id = var_project_storage,
          if_exists='replace')  # Opciones: 'fail', 'replace', 'append'

#### Ejecución SP

In [ ]:
# Llama al procedimiento almacenado con los parámetros
query = f"CALL `{var_project_operation}.raw_stage_dataentry_finanzas.prc_load_table_rpta_rechazo_bbva_izipay`('{var_project_storage}','{var_project_sensitive}','{var_fecha_ini}','{var_fecha_fin}');"
job = clientBQ.query(query)
# Espera a que la consulta se complete
results = job.result()

### INTERBANK

In [ ]:
#@title Variables Banco y Tabla { run: "auto", display-mode: "form" }
var_banco = "interbank" #@param {type:"string"}
var_banco_cod = "MCPR348A" #@param {type:"string"}
var_table = "rpta_rechazo_interbank" #@param {type:"string"}

#### Variables Reproceso

In [ ]:
#revisar archivo 20250312

In [ ]:
# # # Parámetros de fecha
# fecha_inicial = datetime.strptime("2025-03-13", "%Y-%m-%d")
# fecha_final = datetime.strptime("2025-09-25", "%Y-%m-%d")

In [ ]:
fecha_inicial, fecha_final

#### Carga Archivos

In [ ]:
# Crear cliente de Cloud Storage
bucket = client.bucket(bucket_name)

In [ ]:
# Lista para almacenar los DataFrames
dataframes = []

In [ ]:
fecha_actual = fecha_inicial
while fecha_actual <= fecha_final:
    fecha_str = fecha_actual.strftime("%Y%m%d")
    anho_str = fecha_actual.strftime("%Y")
    mes_str = fecha_actual.strftime("%m")
    dia_str = fecha_actual.strftime("%d")

    folder_path = f'Raw/Data_Entry/Finanzas/{var_banco}/{anho_str}/{mes_str}/{dia_str}'
    # --filename = f'MCPR345_{fecha_str}_190301_Rep_operaciones_rechazadas_BBVA_PMP.txt'

    prefijo = f"{folder_path}/{var_banco_cod}_{fecha_str}_"
    blobs = bucket.list_blobs(prefix=prefijo)
    # blobs_filter = [blob for blob in blobs if 'PMP' in blob.name]
    for blob in blobs:

    # Ruta completa dentro del bucket
    # --blob_name = f'{folder_path}/{filename}'
    # print({blob_name})
    # --blob = bucket.blob(blob_name)

      if blob.exists():
              # Leer el contenido del archivo
              # --file_content = blob.download_as_text()
              file_content = blob.download_as_string()

              # Dividir por líneas
              lines = file_content.splitlines()

              # Eliminar las primeras 5 líneas y la línea 7 (índice 6 después de eliminar 5)
              lines_filtered = lines[5:]  # eliminar primeras 5

              # Convertir a un archivo de tipo buffer (simula un archivo real)
              lines_filtered_str = [line.decode("latin1") for line in lines_filtered]

              lines_fixed = [line.replace("A  TIPO DOC.","A     TIPO DOC.") for line in lines_filtered_str]

              buffer = io.StringIO('\n'.join(lines_fixed))
              # --buffer = io.StringIO('\n'.join(lines_filtered))
              df = pd.read_csv(buffer, sep=r'\s{4,}', engine='python', encoding='utf-8', dtype=str)
              df = df[df.isnull().sum(axis=1) < 5].reset_index(drop=True)
              df.columns = [
                  "comercio",
                  "fecha_apertura",
                  "tipo_documento",
                  "nro_documento",
                  "razon_social",
                  "nro_cuenta",
                  "banco",
                  "moneda",
                  "importe",
                  "archivo",
                  "motivo_rechazo"
              ]
              # Agregar las columnas con los valores extraídos del nombre del archivo
              df['process_date'] = fecha_str  # La fecha extraída del nombre del archivo
              dataframes.append(df)
              print(f"Carga Exitosa!!!: {blob}")
      else:
          print(f"Archivo no encontrado en GCS: {blob}")

    fecha_actual += timedelta(days=1)

In [ ]:
dataframes

In [ ]:
if len(dataframes) == 0:
    df_final = pd.DataFrame()  # Devuelve un DataFrame vacío y continúa
elif len(dataframes) == 1:
    df_final = dataframes[0]
else:
    df_final = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_final['process_date'] = pd.to_datetime(df_final['process_date'], format='%Y%m%d')

In [ ]:
print(df_final.dtypes)

#### Carga a Bigquery

In [ ]:
df_final.to_gbq(destination_table=f'raw_stage_dataentry_finanzas.{var_table}',
          project_id = var_project_storage,
          if_exists='replace')  # Opciones: 'fail', 'replace', 'append'

#### Ejecución SP

In [ ]:
# Llama al procedimiento almacenado con los parámetros
query = f"CALL `{var_project_operation}.raw_stage_dataentry_finanzas.prc_load_table_rpta_rechazo_interbank`('{var_project_storage}','{var_project_sensitive}','{var_fecha_ini}','{var_fecha_fin}');"
job = clientBQ.query(query)
# Espera a que la consulta se complete
results = job.result()

### SCOTIABANK

In [ ]:
#@title Variables Banco y Tabla { run: "auto", display-mode: "form" }
var_banco = "scotiabank" #@param {type:"string"}
var_banco_cod = "MCPR351" #@param {type:"string"}
var_table = "rpta_rechazo_scotiabank" #@param {type:"string"}

#### Variables Reproceso

In [ ]:
# # Parámetros de fecha
# fecha_inicial = datetime.strptime("2025-10-01", "%Y-%m-%d")
# fecha_final = datetime.strptime("2025-10-14", "%Y-%m-%d")

In [ ]:
fecha_inicial,fecha_final

#### Carga Archivos

In [ ]:
# Crear cliente de Cloud Storage
bucket = client.bucket(bucket_name)

In [ ]:
# Lista para almacenar los DataFrames
dataframes = []

In [ ]:
folder_path

In [ ]:
fecha_actual = fecha_inicial
while fecha_actual <= fecha_final:
    fecha_str = fecha_actual.strftime("%Y%m%d")
    anho_str = fecha_actual.strftime("%Y")
    mes_str = fecha_actual.strftime("%m")
    dia_str = fecha_actual.strftime("%d")

    folder_path = f'Raw/Data_Entry/Finanzas/{var_banco}/{anho_str}/{mes_str}/{dia_str}'
    # --filename = f'MCPR345_{fecha_str}_190301_Rep_operaciones_rechazadas_BBVA_PMP.txt'

    prefijo = f"{folder_path}/{var_banco_cod}_{fecha_str}"
    blobs = bucket.list_blobs(prefix=prefijo)
    # blobs_filter = [blob for blob in blobs if 'PMP' in blob.name]
    for blob in blobs:

    # Ruta completa dentro del bucket
    # --blob_name = f'{folder_path}/{filename}'
    # print({blob_name})
    # --blob = bucket.blob(blob_name)

      if blob.exists():
              # Leer el contenido del archivo
              # --file_content = blob.download_as_text()
              file_content = blob.download_as_string()

              # Dividir por líneas
              lines = file_content.splitlines()

              # Eliminar las primeras 5 líneas y la línea 7 (índice 6 después de eliminar 5)
              lines_filtered = lines[5:]  # eliminar primeras 5

              # Convertir a un archivo de tipo buffer (simula un archivo real)
              lines_filtered_str = [line.decode("latin1") for line in lines_filtered]

              lines_fixed = [line.replace("A  TIPO DOC.","A     TIPO DOC.") for line in lines_filtered_str]

              buffer = io.StringIO('\n'.join(lines_fixed))
              # --buffer = io.StringIO('\n'.join(lines_filtered))
              df = pd.read_csv(buffer, sep=r'\s{4,}', engine='python', encoding='utf-8', dtype=str)
              df = df[df.isnull().sum(axis=1) < 5].reset_index(drop=True)
              df.columns = [
                  "comercio",
                  "fecha_apertura",
                  "tipo_documento",
                  "nro_documento",
                  "razon_social",
                  "nro_cuenta",
                  "banco",
                  "moneda",
                  "importe",
                  "archivo",
                  "motivo_rechazo"
              ]
              # Agregar las columnas con los valores extraídos del nombre del archivo
              df['process_date'] = fecha_str  # La fecha extraída del nombre del archivo
              dataframes.append(df)
              print(f"Carga Exitosa!!!: {blob}")
      else:
          print(f"Archivo no encontrado en GCS: {blob}")

    fecha_actual += timedelta(days=1)

In [ ]:
if len(dataframes) == 0:
    df_final = pd.DataFrame()  # Devuelve un DataFrame vacío y continúa
elif len(dataframes) == 1:
    df_final = dataframes[0]
else:
    df_final = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_final['process_date'] = pd.to_datetime(df_final['process_date'], format='%Y%m%d')

In [ ]:
print(df_final.dtypes)

#### Carga a Bigquery

In [ ]:
df_final.to_gbq(destination_table=f'raw_stage_dataentry_finanzas.{var_table}',
          project_id = var_project_storage,
          if_exists='replace')  # Opciones: 'fail', 'replace', 'append'

#### Ejecución SP

In [ ]:
# Llama al procedimiento almacenado con los parámetros
query = f"CALL `{var_project_operation}.raw_stage_dataentry_finanzas.prc_load_table_rpta_rechazo_scotiabank`('{var_project_storage}','{var_project_sensitive}','{var_fecha_ini}','{var_fecha_fin}');"
job = clientBQ.query(query)
# Espera a que la consulta se complete
results = job.result()

### BCP

In [ ]:
#@title Variables Banco y Tabla { run: "auto", display-mode: "form" }
var_banco = "bcp" #@param {type:"string"}
var_banco_cod = "MCPR344A" #@param {type:"string"}
var_table = "rpta_rechazo_bcp" #@param {type:"string"}

#### Variables Reproceso

In [ ]:
# # Parámetros de fecha
# fecha_inicial = datetime.strptime("2025-10-01", "%Y-%m-%d")
# fecha_final = datetime.strptime("2025-10-14", "%Y-%m-%d")

In [ ]:
fecha_inicial,fecha_final

#### Carga Archivos

In [ ]:
# Crear cliente de Cloud Storage
bucket = client.bucket(bucket_name)

In [ ]:
# Lista para almacenar los DataFrames
dataframes = []

In [ ]:
fecha_actual = fecha_inicial
while fecha_actual <= fecha_final:
    fecha_str = fecha_actual.strftime("%Y%m%d")
    anho_str = fecha_actual.strftime("%Y")
    mes_str = fecha_actual.strftime("%m")
    dia_str = fecha_actual.strftime("%d")

    folder_path = f'Raw/Data_Entry/Finanzas/{var_banco}/{anho_str}/{mes_str}/{dia_str}'
    # --filename = f'MCPR345_{fecha_str}_190301_Rep_operaciones_rechazadas_BBVA_PMP.txt'

    prefijo = f"{folder_path}/{var_banco_cod}_{fecha_str}"
    blobs = bucket.list_blobs(prefix=prefijo)
    # blobs_filter = [blob for blob in blobs if 'PMP' in blob.name]
    for blob in blobs:

    # Ruta completa dentro del bucket
    # --blob_name = f'{folder_path}/{filename}'
    # print({blob_name})
    # --blob = bucket.blob(blob_name)

      if blob.exists():
              # Leer el contenido del archivo
              # --file_content = blob.download_as_text()
              file_content = blob.download_as_string()

              # Dividir por líneas
              lines = file_content.splitlines()

              # Eliminar las primeras 5 líneas y la línea 7 (índice 6 después de eliminar 5)
              lines_filtered = lines[5:]  # eliminar primeras 5

              # Convertir a un archivo de tipo buffer (simula un archivo real)
              lines_filtered_str = [line.decode("latin1") for line in lines_filtered]

              lines_fixed = [line.replace("A  TIPO DOC.","A     TIPO DOC.") for line in lines_filtered_str]
              lines_fixed_2 = [line.replace("           ERROR"," ERROR") for line in lines_fixed]

              buffer = io.StringIO('\n'.join(lines_fixed_2))
              # --buffer = io.StringIO('\n'.join(lines_filtered))
              df = pd.read_csv(buffer, sep=r'\s{4,}', engine='python', encoding='utf-8', dtype=str)
              df = df[df.isnull().sum(axis=1) < 5].reset_index(drop=True)
              df.columns = [
                  "comercio",
                  "fecha_apertura",
                  "tipo_documento",
                  "nro_documento",
                  "razon_social",
                  "tipo_cuenta",
                  "nro_cuenta",
                  "banco",
                  "moneda",
                  "importe",
                  "archivo",
                  "motivo_rechazo"
              ]
              # Agregar las columnas con los valores extraídos del nombre del archivo
              df['process_date'] = fecha_str  # La fecha extraída del nombre del archivo
              dataframes.append(df)
              print(f"Carga Exitosa!!!: {blob}")
      else:
          print(f"Archivo no encontrado en GCS: {blob}")

    fecha_actual += timedelta(days=1)

In [ ]:
if len(dataframes) == 0:
    df_final = pd.DataFrame()  # Devuelve un DataFrame vacío y continúa
elif len(dataframes) == 1:
    df_final = dataframes[0]
else:
    df_final = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_final['process_date'] = pd.to_datetime(df_final['process_date'], format='%Y%m%d')

In [ ]:
print(df_final.dtypes)

#### Carga a Bigquery

In [ ]:
df_final.to_gbq(destination_table=f'raw_stage_dataentry_finanzas.{var_table}',
          project_id = var_project_storage,
          if_exists='replace')  # Opciones: 'fail', 'replace', 'append'

#### Ejecución SP

In [ ]:
# Llama al procedimiento almacenado con los parámetros
query = f"CALL `{var_project_operation}.raw_stage_dataentry_finanzas.prc_load_table_rpta_rechazo_bcp`('{var_project_storage}','{var_project_sensitive}','{var_fecha_ini}','{var_fecha_fin}');"
job = clientBQ.query(query)
# Espera a que la consulta se complete
results = job.result()